# NB10 — Regresión M3: RF Tuneado + XGBoost (MSE + Poisson)
**ZMM Movilidad Predictiva**

**Objetivo:** Predecir `siniestros_zona_industrial` (conteo continuo) sin overfitting.

**Mejoras:** Hiperparámetros limitados, NaN flag+mediana, XGBoost Poisson para count data.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import xgboost as xgb
import pickle
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline

RUTA_PROCESSED = '../data_processed/'
RUTA_OUTPUTS   = '../outputs/'

print('='*65)
print('NB10: REGRESION M3 — RF TUNEADO + XGBOOST')
print('='*65)

## 1. Carga y Features

In [ ]:
df = pd.read_csv(RUTA_PROCESSED + 'super_tabla_con_clusters.csv', parse_dates=['fecha_hora'])
df = df[(df['fecha_hora'] >= '2023-01-01') & (df['fecha_hora'] <= '2025-12-31 23:00:00')].copy()

# NaN espaciales
spatial_feats = ['dist_industrial_promedio', 'dist_industrial_minima',
                 'siniestros_en_zona', 'masa_laboral_max']
df['tiene_dato_espacial'] = df[spatial_feats[0]].notna().astype(int)
for sf in spatial_feats:
    df[sf] = df[sf].fillna(df[sf].median())

if 'tipo_dia' in df.columns:
    td = pd.get_dummies(df['tipo_dia'], prefix='td', drop_first=True)
    df = pd.concat([df, td], axis=1)
    tipo_dia_cols = td.columns.tolist()
else:
    tipo_dia_cols = []

features_modelo = [
    'hora_del_dia', 'dia_semana', 'es_fin_de_semana', 'intensidad_hora_pico',
    'temperatura_c', 'nivel_lluvia', 'impacto_evento_activo', 'asistencia_estimada',
    'dist_industrial_promedio', 'dist_industrial_minima',
    'siniestros_en_zona', 'masa_laboral_max', 'tiene_dato_espacial', 'cluster_hora',
] + tipo_dia_cols
if 'nivel_impacto' in df.columns:
    features_modelo.append('nivel_impacto')
features_modelo = [f for f in features_modelo if f in df.columns]

X = df[features_modelo]; y = df['siniestros_zona_industrial']
print(f'Shape: {df.shape} | Features: {len(features_modelo)}')
print(f'Target: min={y.min()} max={y.max()} mean={y.mean():.2f}')

## 2. Entrenamiento: RF + XGBoost (MSE + Poisson)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,}')

def eval_reg(name, model, Xtr, ytr, Xte, yte):
    ptr, pte = model.predict(Xtr), model.predict(Xte)
    r2tr, r2te = r2_score(ytr, ptr), r2_score(yte, pte)
    rmse = np.sqrt(mean_squared_error(yte, pte))
    mae = mean_absolute_error(yte, pte)
    print(f'{name:<20} R2 train={r2tr:.4f} test={r2te:.4f} (gap={r2tr-r2te:.4f}) RMSE={rmse:.4f} MAE={mae:.4f}')
    return r2te, rmse, mae, pte

# RF
rf = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=10,
    min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)
rf_r2, rf_rmse, rf_mae, rf_pred = eval_reg('RF Tuneado', rf, X_train, y_train, X_test, y_test)

# XGB MSE
xm = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    objective='reg:squarederror', early_stopping_rounds=30)
xm.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xm_r2, xm_rmse, xm_mae, xm_pred = eval_reg('XGB MSE', xm, X_train, y_train, X_test, y_test)

# XGB Poisson
xp = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
    subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
    objective='count:poisson', early_stopping_rounds=30)
xp.fit(X_train, y_train, eval_set=[(X_test, y_test)], verbose=False)
xp_r2, xp_rmse, xp_mae, xp_pred = eval_reg('XGB Poisson', xp, X_train, y_train, X_test, y_test)

# Ganador
modelos = {'RF Tuneado':(rf,rf_r2,rf_rmse,rf_mae,rf_pred),
           'XGB MSE':(xm,xm_r2,xm_rmse,xm_mae,xm_pred),
           'XGB Poisson':(xp,xp_r2,xp_rmse,xp_mae,xp_pred)}
ganador = max(modelos, key=lambda k: modelos[k][1])
gm, gr2, grmse, gmae, gpred = modelos[ganador]
print(f'\nGANADOR: {ganador} (R2={gr2:.4f})')

## 3. Validación Temporal + Visualizaciones

In [ ]:
# Temporal
dtt = df[df['fecha_hora'].dt.year.isin([2023,2024])]
dtv = df[df['fecha_hora'].dt.year == 2025]
if 'XGB' in ganador:
    obj = 'count:poisson' if 'Poisson' in ganador else 'reg:squarederror'
    tm = xgb.XGBRegressor(n_estimators=300, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=42, n_jobs=-1,
        objective=obj, early_stopping_rounds=30)
    tm.fit(dtt[features_modelo], dtt['siniestros_zona_industrial'],
           eval_set=[(dtv[features_modelo], dtv['siniestros_zona_industrial'])], verbose=False)
else:
    tm = RandomForestRegressor(n_estimators=200, max_depth=15, min_samples_split=10,
        min_samples_leaf=5, max_features='sqrt', random_state=42, n_jobs=-1)
    tm.fit(dtt[features_modelo], dtt['siniestros_zona_industrial'])
temp_r2 = r2_score(dtv['siniestros_zona_industrial'], tm.predict(dtv[features_modelo]))
print(f'Temporal R2: {temp_r2:.4f}')

# Scatter
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
axes[0].scatter(y_train, gm.predict(X_train), alpha=0.1, s=1)
axes[0].plot([0,y.max()],[0,y.max()],'r--',lw=2); axes[0].set_title('Train'); axes[0].grid(True,alpha=0.3)
axes[0].set_xlabel('Real'); axes[0].set_ylabel('Predicho')
axes[1].scatter(y_test, gpred, alpha=0.1, s=1)
axes[1].plot([0,y.max()],[0,y.max()],'r--',lw=2); axes[1].set_title(f'Test (R2={gr2:.3f})'); axes[1].grid(True,alpha=0.3)
axes[1].set_xlabel('Real'); axes[1].set_ylabel('Predicho')
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'm3_real_vs_predicho.png', dpi=150); plt.show()

# Feature importance
imp_df = pd.DataFrame({'feature': features_modelo,
    'importance': gm.feature_importances_}).sort_values('importance', ascending=False)
fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=imp_df.head(15), y='feature', x='importance', palette='plasma', ax=ax)
ax.set_title(f'Feature Importance — M3 ({ganador})')
plt.tight_layout(); plt.savefig(RUTA_OUTPUTS + 'm3_feature_importance.png', dpi=150); plt.show()

In [ ]:
# Guardar
modelo_data = {
    'modelo': gm, 'nombre': ganador, 'features': features_modelo,
    'target': 'siniestros_zona_industrial',
    'rmse_test': grmse, 'mae_test': gmae, 'r2_test': gr2,
    'feature_importance': imp_df, 'temporal_r2': temp_r2,
}
with open(RUTA_OUTPUTS + 'modelo_m3_rf.pkl', 'wb') as f:
    pickle.dump(modelo_data, f)
print(f'Modelo guardado: modelo_m3_rf.pkl ({ganador})')
print(f'R2 test: {gr2:.4f} | R2 temporal: {temp_r2:.4f}')